# Primer análisis de partidos de Champions

Este notebook explora la copia incluida de OpenFootball. **Todavía no entrena un modelo ni produce predicciones.**

Fuente: https://github.com/openfootball/champions-league. Licencia CC0-1.0. Versión exacta y hashes en `data/fuentes.json`.

En Jupyter, abre el notebook dentro del proyecto. En Google Colab, sube este notebook y, cuando se solicite, el archivo `data/processed/partidos.csv` del proyecto.


In [1]:
from pathlib import Path
import pandas as pd

candidatos = [Path("../data/processed/partidos.csv"), Path("data/processed/partidos.csv"), Path("partidos.csv")]
ruta = next((p for p in candidatos if p.exists()), None)
if ruta is None:
    from google.colab import files
    print("Sube el archivo partidos.csv incluido en el proyecto.")
    archivos = files.upload()
    ruta = next((Path(nombre) for nombre in archivos if nombre.endswith(".csv")), None)
    if ruta is None:
        raise ValueError("Debes subir partidos.csv")
df = pd.read_csv(ruta, encoding="utf-8-sig", parse_dates=["fecha"])
print("Partidos:", len(df))
print("Equipos:", len(set(df.local) | set(df.visitante)))
print("Fechas:", df.fecha.min().date(), "a", df.fecha.max().date())


Partidos: 994
Equipos: 83
Fechas: 2016-09-13 a 2024-06-01


## 1. Estructura y controles iniciales
Cada fila representa un partido. Los goles y la etiqueta corresponden a los 90 minutos más tiempo añadido.


In [2]:
print(df[["fecha", "local", "visitante", "goles_local_90", "goles_visitante_90", "resultado"]].head().to_string(index=False))
print("\nFaltantes por columna:\n", df.isna().sum().to_string())
print("\nIDs duplicados:", df.id.duplicated().sum())


     fecha          local              visitante  goles_local_90  goles_visitante_90 resultado
2016-09-13 Bayern München              FK Rostov               5                   0     Local
2016-09-13    Dinamo Kiev             SSC Napoli               1                   2 Visitante
2016-09-13   FC Barcelona              Celtic FC               7                   0     Local
2016-09-13  FC Basel 1893 PFC Ludogorets Razgrad               1                   1    Empate
2016-09-13  PSV Eindhoven        Atlético Madrid               0                   1 Visitante

Faltantes por columna:
 id                    0
temporada             0
fecha                 0
fase                  0
local                 0
visitante             0
pais_local            0
pais_visitante        0
goles_local_90        0
goles_visitante_90    0
resultado             0
hubo_prorroga         0
hubo_penales          0
local_original        0
visitante_original    0
marcador_original     0
fuente               

## 2. Cobertura y distribución
La temporada 2019/20 contiene 119 partidos en esta fuente. Las otras seleccionadas contienen 125. Los conteos se verifican contra el encabezado de cada archivo original.


In [3]:
print(df.groupby("temporada").size().rename("partidos").to_string())
resumen = df.resultado.value_counts().reindex(["Local", "Empate", "Visitante"]).rename("partidos").to_frame()
resumen["porcentaje"] = (100 * resumen.partidos / len(df)).round(2)
print("\n", resumen.to_string())
print("\nPromedio de goles a 90 min:", round((df.goles_local_90 + df.goles_visitante_90).mean(), 2))


temporada
2016-17    125
2017-18    125
2018-19    125
2019-20    119
2020-21    125
2021-22    125
2022-23    125
2023-24    125

            partidos  porcentaje
resultado                      
Local           448       45.07
Empate          212       21.33
Visitante       334       33.60

Promedio de goles a 90 min: 3.03


## 3. Tratamiento de prórrogas
El marcador original y el resultado de los 90 minutos se conservan por separado. Una tanda de penales no determina la clase de resultado a 90 minutos.


In [4]:
print(df.loc[df.hubo_prorroga, ["fecha", "local", "visitante", "marcador_original", "goles_local_90", "goles_visitante_90", "resultado"]].to_string(index=False))


     fecha           local       visitante              marcador_original  goles_local_90  goles_visitante_90 resultado
2017-04-18     Real Madrid  Bayern München          4-2 a.e.t. (1-2, 0-0)               1                   2 Visitante
2019-03-06        FC Porto         AS Roma          3-1 a.e.t. (2-1, 1-1)               2                   1     Local
2020-03-11    Liverpool FC Atlético Madrid          2-3 a.e.t. (1-0, 1-0)               1                   0     Local
2021-03-09        Juventus        FC Porto          3-2 a.e.t. (2-1, 0-1)               2                   1     Local
2022-04-12     Real Madrid      Chelsea FC          2-3 a.e.t. (1-3, 0-1)               1                   3 Visitante
2022-05-04     Real Madrid Manchester City          3-1 a.e.t. (2-1, 0-0)               2                   1     Local
2024-03-12      Arsenal FC        FC Porto 4-2 pen. 1-0 a.e.t. (1-0, 1-0)               1                   0     Local
2024-03-13 Atlético Madrid           Int

## 4. Ejemplo de variables anteriores al partido
Se filtra por fecha estrictamente anterior al corte antes de seleccionar los últimos cinco partidos. El historial se limita a Champions. No se interpreta como una predicción.


In [5]:
equipo = "Real Madrid"
corte = pd.Timestamp("2024-04-09")
historial = df[(df.fecha < corte) & ((df.local == equipo) | (df.visitante == equipo))].sort_values("fecha").tail(5).copy()
historial["goles_favor"] = historial.apply(lambda r: r.goles_local_90 if r.local == equipo else r.goles_visitante_90, axis=1)
historial["goles_contra"] = historial.apply(lambda r: r.goles_visitante_90 if r.local == equipo else r.goles_local_90, axis=1)
print(historial[["fecha", "local", "visitante", "goles_favor", "goles_contra"]].to_string(index=False))
print("\nPartidos usados:", len(historial))
print("Promedio de goles a favor:", historial.goles_favor.mean())
print("Promedio de goles en contra:", historial.goles_contra.mean())
assert (historial.fecha < corte).all()


     fecha              local               visitante  goles_favor  goles_contra
2023-11-08        Real Madrid Sporting Clube de Braga            3             0
2023-11-29        Real Madrid              SSC Napoli            4             2
2023-12-12 1. FC Union Berlin             Real Madrid            3             2
2024-02-13         RB Leipzig             Real Madrid            1             0
2024-03-06        Real Madrid              RB Leipzig            1             1

Partidos usados: 5
Promedio de goles a favor: 2.4
Promedio de goles en contra: 1.0


## 5. Plan de la siguiente etapa

- Construir estas variables históricas para cada partido, excluyendo su propio resultado.
- Definir qué hacer cuando un equipo tiene poco historial y con los encuentros en sedes neutrales.
- Entrenamiento: 2016/17 a 2020/21. Validación: 2021/22 y 2022/23. Prueba final: 2023/24.
- Comparar una referencia de clase mayoritaria, regresión logística y Random Forest.
- Medir accuracy, F1 macro, matriz de confusión y log loss para las probabilidades.
- Ajustar cualquier transformación solo con los datos de entrenamiento.

**Limitaciones:** fuente comunitaria sin auditoría independiente de cada resultado, datos solo hasta junio de 2024, ausencia de lesiones y alineaciones, y falta de un campo uniforme de sede neutral. Los goles del partido objetivo sirven para crear su etiqueta, no como entrada del modelo.
